Three mechanisms that show up everywhere in real Python code — and let you process large datasets without exhausting memory.

::: {.callout-tip}
## Why learn this?
These three tools are how professional Python stays fast and memory-light: add timing or caching *without touching your logic*, and process data far bigger than your RAM.

- **Imagine you need** to time every expensive routine — one `@timed` on top, no edits inside the function.
- **Imagine you need** to process a 50 GB velocity time series — a *generator* streams it in chunks so you never load it all at once.
:::

## Decorators
A decorator is a function that wraps another function to add behaviour (timing, logging, caching, validation) without editing it. `@wraps` preserves the wrapped function's name and docstring.

In [1]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        dt = time.perf_counter() - t0
        print(f'{func.__name__} took {dt*1e3:.2f} ms')
        return result
    return wrapper

@timed
def slow_sum(n):
    """Sum 0..n-1."""
    return sum(range(n))

print('result:', slow_sum(1_000_00))
print('name kept by @wraps:', slow_sum.__name__)

slow_sum took 1.36 ms
result: 4999950000
name kept by @wraps: slow_sum


### Decorators that take arguments
Add one more layer: a function that *returns* a decorator.

In [2]:
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def greet():
    print('hello')

greet()

# A very useful built-in decorator: memoisation
from functools import lru_cache
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print('fib(30):', fib(30))

hello
hello
hello
fib(30): 832040


## Iterators & iterables
An **iterable** can produce an **iterator** (via `iter()`); an iterator yields items one at a time (via `next()`) and raises `StopIteration` when done. `for` loops use this protocol under the hood.

In [3]:
nums = [10, 20, 30]          # a list is iterable
it = iter(nums)             # get an iterator
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print('exhausted')

# A custom iterator: countdown
class Countdown:
    def __init__(self, start):
        self.n = start
    def __iter__(self):
        return self
    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

print(list(Countdown(4)))

10 20 30
exhausted
[4, 3, 2, 1]


**Imagine you need** to compute the mean of a velocity record too large to fit in memory. This is super easy in Python using a *generator* that yields one chunk at a time.

## Generators
A `generator` produces values lazily with `yield` — it computes each item on demand, so it uses almost no memory even for huge or infinite sequences. Ideal for streaming through large simulation output.

In [4]:
def read_in_chunks(seq, size):
    """Yield successive size-length chunks — nothing is materialised whole."""
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

for chunk in read_in_chunks(list(range(10)), 3):
    print(chunk)

# generator EXPRESSION: like a comprehension but lazy (parentheses)
squares = (n * n for n in range(1_000_000))
print('first three:', next(squares), next(squares), next(squares))
print('type:', type(squares).__name__)

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9]
first three: 0 1 4
type: generator


In [5]:
# itertools: composable, memory-light iterator building blocks
import itertools as it

print('count :', list(it.islice(it.count(0, 5), 4)))     # 0,5,10,15
print('cycle :', list(it.islice(it.cycle('AB'), 5)))     # A,B,A,B,A
print('chain :', list(it.chain([1, 2], [3, 4])))
print('pairs :', list(it.combinations('xyz', 2)))

count : [0, 5, 10, 15]
cycle : ['A', 'B', 'A', 'B', 'A']
chain : [1, 2, 3, 4]
pairs : [('x', 'y'), ('x', 'z'), ('y', 'z')]


## Self-tests

**1.** Write a decorator `announce` that prints `calling <name>` before the function runs, then returns its result.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
from functools import wraps
def announce(func):
    @wraps(func)
    def wrapper(*a, **k):
        print('calling', func.__name__)
        return func(*a, **k)
    return wrapper

@announce
def add(a, b):
    return a + b
print(add(2, 3))

**2.** Write a **generator** `evens(limit)` yielding even numbers `0, 2, 4, …` below `limit`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
def evens(limit):
    n = 0
    while n < limit:
        yield n
        n += 2
print(list(evens(10)))

## Turbulence in practice: streaming statistics

A long velocity record — or a stack of DNS snapshots — may not fit in memory. A **generator** streams it chunk by chunk, and Welford's algorithm accumulates the mean and variance in a single pass with $O(1)$ memory.

In [8]:
import numpy as np

def stream(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

signal = 10 + np.random.default_rng(0).standard_normal(100_000)  # 'huge'

n = 0
mean = 0.0
M2 = 0.0
for chunk in stream(signal, 1000):
    for x in chunk:
        n += 1
        delta = x - mean
        mean += delta / n
        M2 += delta * (x - mean)
print(f'<u> = {mean:.4f},  u_rms = {(M2 / n) ** 0.5:.4f}  (one pass)')

<u> = 9.9991,  u_rms = 1.0001  (one pass)


**Self-test.** Write a generator `increments(seq, r)` that yields $u(i{+}r)-u(i)$ — the building block of a structure function.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
def increments(seq, r):
    for i in range(len(seq) - r):
        yield seq[i + r] - seq[i]

print(list(increments([0, 1, 4, 9, 16], r=1)))